# Physical Woodwind Synthesis — Complete Component

One self-contained notebook for the whole woodwind track (milestones M0–M5): a
physical-model synthesiser for reed and jet woodwinds, built from digital
waveguides. Nothing here is a sample or an oscillator bank — it simulates the air,
the tube, the reed and the jet, so the timbre (odd-harmonic clarinet, full-series
flute, register jumps) *emerges* from the physics.

**Signal chain:** noise/reed/jet source → two travelling-wave delay lines →
scattering junctions (bore shape) → frequency-dependent loss → radiation filter at
the bell → the sound you hear.

**Every stage is validated by an impulse test** (feed one sample, FFT the output,
confirm the resonances land where theory predicts) *before* a nonlinear source is
trusted on top of it. Run the cells top to bottom.

Layout: building blocks → M0 Karplus–Strong → M1/M2 the tube (+ the octave/odd
correction) → M3 bore shape (cylinder vs cone) → M4 reed instrument → M5 radiation
→ jet flute → instrument presets → tuning guide → honest limitations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

SR = 44100      # sample rate
C  = 343.0      # speed of sound, m/s

## Building blocks

Small stateful nodes, one physical job each. The same discipline the Rust/WASM port
needs: state lives in the node, one difference equation per `tick`.

In [ ]:
class DelayLine:
    """A digital waveguide is two of these — the two travelling-wave halves of a 1-D
    tube (d'Alembert). Fractional delay (linear interpolation) is built in from the
    start: integer-only delay quantises pitch to a grid far coarser than the ear's
    ~0.3% JND, so notes come out audibly out of tune. read()/write() are separate so
    a chain of sections can update in lock-step without an instantaneous loop."""
    def __init__(self, delay, headroom=4):
        self.int_d = int(np.floor(delay))
        self.frac  = delay - self.int_d
        self.size  = self.int_d + headroom
        self.buf   = np.zeros(self.size)
        self.w     = 0
    def read(self):
        i0 = (self.w - self.int_d) % self.size
        i1 = (self.w - self.int_d - 1) % self.size
        return self.buf[i0]*(1.0-self.frac) + self.buf[i1]*self.frac
    def write(self, x):
        self.buf[self.w] = x
        self.w = (self.w + 1) % self.size

class OnePole:
    """One-pole low-pass y[n]=(1-a)x[n]+a y[n-1]. Two uses: frequency-dependent loss
    (real tubes lose highs faster than lows -> warm decay), and the radiation
    reflectance at the bell. Larger a -> lower cutoff -> more high-frequency loss."""
    def __init__(self, a): self.a = a; self.y = 0.0
    def tick(self, x):
        self.y = (1.0 - self.a) * x + self.a * self.y
        return self.y

class DCBlock:
    """One-pole high-pass at a few Hz. Nonlinear sources accumulate a DC offset that
    wastes headroom and thumps speakers; strip it on the output."""
    def __init__(self, r=0.995): self.r=r; self.x1=0.0; self.y1=0.0
    def tick(self, x):
        y = x - self.x1 + self.r*self.y1; self.x1 = x; self.y1 = y; return y

def normalize(x):
    p = np.max(np.abs(x)); return x/p if p > 0 else x

def cutoff_from_radius(radius_m):
    """Radiation cutoff from opening size (ka ~ 1): bigger mouth -> lower cutoff ->
    lower frequencies escape. The physical meaning of a bell."""
    return C / (2*np.pi*radius_m)

## Analysis toolkit

You cannot tune what you cannot see. The impulse test + FFT is the workhorse: it
turns a wiggly time-domain buffer into a picture of *which frequencies are present*,
which is the only way to check the tube against theory.

In [ ]:
def spectrum_db(sig, sr=SR):
    w = np.hanning(len(sig))
    mag = np.abs(np.fft.rfft(sig*w))
    return np.fft.rfftfreq(len(sig), 1/sr), 20*np.log10(mag/np.max(mag) + 1e-6)

def find_peaks(sig, sr=SR, top=10, thr=0.04):
    w = np.hanning(len(sig)); S = np.abs(np.fft.rfft(sig*w))
    f = np.fft.rfftfreq(len(sig), 1/sr)
    idx = [k for k in range(2, len(S)-2)
           if S[k] > S[k-1] and S[k] > S[k+1] and S[k] > thr*S.max()]
    idx = sorted(idx, key=lambda k: -S[k])[:top]
    return sorted(round(float(f[k]), 1) for k in idx)

def centroid(sig, sr=SR):
    w = np.hanning(len(sig)); S = np.abs(np.fft.rfft(sig*w))
    f = np.fft.rfftfreq(len(sig), 1/sr)
    return round(float(np.sum(f*S)/np.sum(S)), 0)

def plot_spectra(pairs, f0=None, xlim=3500, title=""):
    """pairs: list of (signal, label, color)."""
    plt.figure(figsize=(10, 4.5))
    for sig, label, color in pairs:
        f, db = spectrum_db(sig)
        plt.plot(f, db, label=label, color=color, lw=1.2)
    if f0:
        for k in range(1, 20):
            if f0*k < xlim:
                plt.axvline(f0*k, color="k", ls=":", lw=0.5, alpha=0.3)
    plt.xlim(0, xlim); plt.ylim(-60, 2)
    plt.xlabel("Hz"); plt.ylabel("dB (normalised)"); plt.title(title)
    plt.legend(); plt.tight_layout(); plt.show()

## M0 — Karplus–Strong: the minimal waveguide

A delay line with a one-zero low-pass in its feedback, kicked once by a noise burst.
This *is* the smallest digital waveguide: it's the sanity check that delay-line +
loss + feedback is wired correctly before any physical source goes in. It produces a
plucked-string tone with all harmonics, decaying (highs die first — that's the
low-pass in the loop).

In [ ]:
def karplus_strong(f0=220, dur=1.5, sr=SR, seed=1):
    L = int(round(sr/f0))
    rng = np.random.default_rng(seed)
    buf = list(rng.uniform(-1, 1, L))     # noise burst fills the delay line
    out = np.zeros(int(dur*sr)); y1 = 0.0
    for i in range(len(out)):
        v = buf.pop(0)
        nv = 0.5*(v + y1)                 # one-zero low-pass in the feedback
        y1 = v
        buf.append(nv)
        out[i] = v
    return out

ks = karplus_strong(220)
print("Karplus-Strong peaks:", find_peaks(ks)[:5], " (all harmonics of 220)")
display(Audio(normalize(ks), rate=SR))

## M1 / M2 — The tube, boundaries, and the octave/odd correction

Two delay lines carry the right- and left-going waves; they only interact at the
ends. The **reflection sign** at each end is the whole story:

* open end inverts pressure (`r = -1`), closed/rigid end does not (`r = +1`);
* **open–open** (both invert → net no inversion) repeats every round trip → **all
  harmonics**, fundamental `c/2L`;
* **closed–open** (one inversion) needs *two* round trips to repeat → **odd
  harmonics only**, fundamental `c/4L` — an **octave lower** for the same length.

That octave drop is the real clarinet-vs-flute relationship, and it corrects a
common misconception (changing the boundary does *not* keep the pitch fixed while
only adding harmonics). We confirm it with an impulse test on a linear resonator
(no source), which is the validation gate for everything downstream.

In [ ]:
def resonator_impulse(areas, D_line, mouth_refl, dur=0.5, sr=SR,
                       loss_a=0.05, bell_refl=-0.999):
    """Linear resonator (no reed) for the validation gate. mouth_refl = +1 closed,
    -1 open. Feeds a single-sample impulse; the output spectrum IS the tube's
    resonances. D_line = total one-way delay in samples (fixes the length)."""
    areas = np.asarray(areas, float); N = len(areas); D_sec = D_line/N
    right = [DelayLine(D_sec) for _ in range(N)]
    left  = [DelayLine(D_sec) for _ in range(N)]
    rj = [(areas[k]-areas[k+1])/(areas[k]+areas[k+1]) for k in range(N-1)]
    lp = OnePole(loss_a); n = int(dur*sr); out = np.zeros(n)
    for i in range(n):
        aR = [right[k].read() for k in range(N)]
        aL = [left[k].read()  for k in range(N)]
        inj = mouth_refl*aL[0] + (1.0 if i == 0 else 0.0)
        ri = [0.0]*N; li = [0.0]*N; ri[0] = inj
        for k in range(N-1):
            d = rj[k]*(aR[k]-aL[k+1]); ri[k+1] = aR[k]+d; li[k] = aL[k+1]+d
        li[N-1] = bell_refl*lp.tick(aR[N-1])
        for k in range(N): right[k].write(ri[k]); left[k].write(li[k])
        out[i] = aR[N-1]
    return out

N = 16
A_bore = np.pi*0.007**2
cyl = np.ones(N)*A_bore

oo = resonator_impulse(cyl, 100, mouth_refl=-0.999)   # open-open
co = resonator_impulse(cyl, 100, mouth_refl=+0.999)   # closed-open
print("open-open   peaks:", find_peaks(oo)[:6], " -> all harmonics, f0 ~220")
print("closed-open peaks:", find_peaks(co)[:6], " -> odd harmonics, f0 ~110 (octave down)")
plot_spectra([(oo,"open-open (flute-like): all harmonics","C0"),
              (co,"closed-open (clarinet-like): odd only","C3")],
             xlim=2500, title="Same length, different boundary: octave + harmonic content")

## M3 — Bore shape: cylinder vs cone

Bore shape decides harmonic content. A **cylinder** closed at one end gives odd
harmonics (clarinet). A **cone** expanding from the tip behaves as if both ends were
open — it restores the **even** harmonics too (saxophone, oboe). We approximate an
arbitrary bore as a cascade of short cylindrical sections with a scattering junction
between each (`r = (A_k − A_{k+1})/(A_k + A_{k+1})`).

**Honest result (verify below).** The stepped cone *does* fill in the even harmonics
— its modes are roughly evenly spaced, unlike the cylinder's odd-only comb — but the
low modes are mistuned: a truncated cone (no apex, which is unavoidable here) has
stretched low resonances. This is a real limitation of the cylindrical-section
approximation, not a bug. Getting a sax/oboe properly in tune needs a conical
waveguide with an apex/mouthpiece correction — a genuine extension beyond this
milestone.

In [ ]:
cone = np.linspace(A_bore, A_bore*6, N)   # linearly expanding area

cyl_co  = resonator_impulse(cyl,  100, mouth_refl=+0.999)
cone_co = resonator_impulse(cone, 100, mouth_refl=+0.999)
print("cylinder closed-open:", find_peaks(cyl_co)[:7],  " -> odd only")
print("cone     closed-mouth:", find_peaks(cone_co)[:7], " -> even harmonics fill in, low modes mistuned")
plot_spectra([(cyl_co,"cylinder: odd comb","C3"),
              (cone_co,"cone: even harmonics appear","C2")],
             xlim=2500, title="Cone restores even harmonics (low-mode tuning is imperfect)")

## M4 / M5 — The reed instrument, with radiation

Now the tube stops ringing-and-dying and becomes an instrument. The reed is a
**pressure-controlled valve**: the pressure difference across it sets a reflection
coefficient that **snaps to +1 (closed) above a threshold** — that hard `clip` is
the entire nonlinearity, and it manufactures the odd harmonics that reinforce the
cylinder's odd resonances. Every sample the reed re-injects energy based on the
current bore pressure, so the loop self-sustains and locks to the tube's pitch.

M5 folds in **radiation**: the open end is a filter, not a mirror. Low frequencies
reflect (and sustain the tone); high frequencies escape. The **radiated** output —
what you hear — is a high-passed version of the internal bore pressure, so it's
brighter than the field inside. The engine below runs the reed at the mouth and the
radiation filter at the bell, over a multi-section bore (so you can pass any
`areas`).

In [ ]:
def wind(areas, f0=220, pm=0.5, dur=2.0, sr=SR,
         rad_cutoff_hz=2500, loop_comp=1.1, loss=0.995,
         reed_offset=0.7, reed_slope=-0.44, excite_impulse=False):
    """Reed woodwind. Returns (radiated, internal): what escapes, and the internal
    bore pressure. Cylinder areas -> clarinet. loop_comp trims the end-correction
    delay; retune it if you change rad_cutoff_hz or the bore (see tuning guide)."""
    areas = np.asarray(areas, float); N = len(areas)
    a_rad = float(np.exp(-2*np.pi*rad_cutoff_hz/sr))
    # closed-open cylinder: net inversion per round trip -> f0 = sr/(4*D_line)
    D_line = sr/(4.0*f0) - loop_comp
    D_sec  = D_line/N
    right = [DelayLine(D_sec) for _ in range(N)]
    left  = [DelayLine(D_sec) for _ in range(N)]
    rj = [(areas[k]-areas[k+1])/(areas[k]+areas[k+1]) for k in range(N-1)]
    Rf = OnePole(a_rad); dci = DCBlock(); dco = DCBlock()
    n = int(dur*sr); rad_buf = np.zeros(n); int_buf = np.zeros(n)
    for i in range(n):
        aR = [right[k].read() for k in range(N)]
        aL = [left[k].read()  for k in range(N)]
        bore = aL[0]
        if excite_impulse:
            inj = bore + (1.0 if i == 0 else 0.0)
        else:
            pd = bore - pm
            r = reed_offset + reed_slope*pd
            r = 1.0 if r > 1 else (-1.0 if r < -1 else r)   # hard reed closure
            inj = pm + r*pd
        ri = [0.0]*N; li = [0.0]*N; ri[0] = inj
        for k in range(N-1):
            d = rj[k]*(aR[k]-aL[k+1]); ri[k+1] = aR[k]+d; li[k] = aL[k+1]+d
        arrived = aR[N-1]
        reflected = -loss*Rf.tick(arrived)      # low-pass + inversion -> stays in
        radiated  = arrived + reflected          # complement (high-pass) -> escapes
        li[N-1] = reflected
        for k in range(N): right[k].write(ri[k]); left[k].write(li[k])
        int_buf[i] = dci.tick(bore); rad_buf[i] = dco.tick(radiated)
    return rad_buf, int_buf

rad, internal = wind(cyl, f0=220, pm=0.5, dur=3.0)
print("clarinet radiated peaks:", find_peaks(rad[SR//4:])[:8], " (odd series)")
print("centroid  internal:", centroid(internal[SR//4:]), " radiated:", centroid(rad[SR//4:]),
      " (radiated is brighter)")
display(Audio(normalize(rad), rate=SR))
plot_spectra([(internal[SR//4:],"internal bore pressure","C0"),
              (rad[SR//4:],"radiated (what you hear)","C3")],
             f0=220, title="Clarinet: odd harmonics, radiated tilted toward the highs")

## The lock-in range (register behaviour)

Sweep the breath pressure `pm`. Below a threshold the losses win and nothing sounds;
in the lock-in range the loop entrains to the tube's lowest resonance and the pitch
is rock-solid; push too hard and the reed chokes shut. You never programmed these
regimes — they fall out of the coupled loop.

(Note: a real clarinet's overblow to the *twelfth* needs a register vent that
suppresses the fundamental — a tone-hole effect — which this fixed-bore model
doesn't include; here you get threshold and choke, not a spontaneous register jump.)

In [ ]:
print(f"{'breath pm':>10} {'tail rms':>10}  regime")
for pm in [0.30, 0.40, 0.50, 0.60, 0.70]:
    r, _ = wind(cyl, f0=220, pm=pm, dur=1.5)
    tail = r[-SR//10:]
    rms = float(np.sqrt(np.mean(tail**2)))
    regime = "silent" if rms < 1e-3 else ("SUSTAINS" if rms > 0.05 else "weak/choking")
    print(f"{pm:>10.2f} {rms:>10.3f}  {regime}")

## Jet woodwind — the flute

The other woodwind excitation. A turbulent air jet is deflected across the edge of
the embouchure hole by the tube's own acoustic field; the deflection saturates
(a cubic/`tanh` nonlinearity) and there's an **embouchure delay** for the jet to
travel from the flue to the edge. The tube is **open–open**, so the flute has the
full harmonic series, and its fundamental is an octave above a clarinet of the same
length.

Two behaviours here are physically correct and emergent: the flute self-oscillates
from a *steady* breath (the jet supplies the negative resistance), and it
**overblows to the octave** when you push the breath past the low-register window —
that's how a flutist plays the upper octave.

In [ ]:
def flute(f0=440, pm=1.0, dur=2.0, sr=SR, jet_ratio=0.24, loss_a=0.10,
          jet_refl=0.5, end_refl=0.5, noise=0.03, length_comp=1.0, seed=1):
    """Jet flute. pm ~0.85-1.1 sits in the low register; lower or much higher breath
    overblows to the octave. jet_ratio (embouchure delay / bore) selects the
    register and colours the tone."""
    Db = sr/f0 + length_comp               # open-open: round trip = Db, f0 = sr/Db
    bore = DelayLine(Db)
    jet  = DelayLine(max(jet_ratio*Db, 1.0))
    lp = OnePole(loss_a); dc = DCBlock()
    rng = np.random.default_rng(seed); n = int(dur*sr); out = np.zeros(n)
    for i in range(n):
        p_ret = lp.tick(bore.read())                 # pressure reflected back to mouth
        drive = pm + noise*rng.uniform(-1, 1) - jet_refl*p_ret
        jet.write(drive)
        js = jet.read()
        jflow = js*(js*js - 1.0)                      # cubic jet nonlinearity
        jflow = 1.0 if jflow > 1 else (-1.0 if jflow < -1 else jflow)
        bore.write(jflow + end_refl*p_ret)
        out[i] = dc.tick(p_ret)
    return out

fl = flute(440, pm=1.0, dur=2.0)
print("flute peaks:", find_peaks(fl[SR//4:])[:6], " (all harmonics of 440)")
display(Audio(normalize(fl), rate=SR))
plot_spectra([(fl[SR//4:], "flute (jet, open-open)", "C4")],
             f0=440, xlim=4000, title="Flute: full harmonic series")

print("\noverblow — raise the breath and the register jumps to the octave:")
for pm in [0.5, 1.0, 1.5]:
    o = flute(440, pm=pm, dur=1.2)
    print(f"  pm={pm}: fundamental ~ {find_peaks(o[SR//4:])[:1]}")

## Instrument presets & parameter guide

The *same* engines play different instruments by changing bore shape, source, and a
few coefficients. Presets below are the validated starting points; the tuning guide
that follows explains how to move them.

| instrument | engine | bore | source | pitch rule | notes |
|---|---|---|---|---|---|
| clarinet | `wind` | cylinder | reed (clip) | `f0 = sr/4L`, odd harmonics | woody, hollow; `pm` 0.4–0.65 |
| saxophone / oboe | `wind` | cone (flared areas) | reed (clip) | ~all harmonics | **experimental** — low-mode tuning imperfect (truncated cone) |
| flute | `flute` | cylinder | jet (cubic) | `f0 = sr/2L`, all harmonics | breathy; overblows to octave |

**Per-parameter meaning**

* `areas` — the bore profile. Uniform = cylinder (odd harmonics). Linearly
  increasing = cone (fills in even harmonics). Widen the last few sections for a bell.
* `pm` — breath / embouchure pressure. Reed: 0.4–0.65 sustains, higher chokes.
  Flute: ~0.85–1.1 for the low register, outside that overblows.
* `rad_cutoff_hz` — radiation cutoff = **opening size** (`cutoff_from_radius`).
  Higher = smaller opening = thinner, brighter radiated tone; lower = larger opening
  = fuller. Changing it detunes slightly (end correction) — retrim `loop_comp`.
* `reed_offset`, `reed_slope` — the reed table. `offset` ~ how open at rest;
  `slope` (negative) ~ how fast it closes. More negative slope = brighter/buzzier.
* `jet_ratio` — embouchure delay ÷ bore. Selects the flute register and colour;
  ~0.24 for the low register here.
* `loss` / `loss_a` — round-trip damping. More loss = warmer, shorter, more stable,
  but too much and the reed won't start.

In [ ]:
PRESETS = {
    "clarinet": dict(engine="wind", areas=np.ones(16)*np.pi*0.007**2,
                     f0=220, pm=0.5, rad_cutoff_hz=2500, loop_comp=1.1),
    "sax_like": dict(engine="wind", areas=np.linspace(np.pi*0.007**2, np.pi*0.007**2*6, 16),
                     f0=220, pm=0.5, rad_cutoff_hz=2500, loop_comp=1.1),   # experimental
    "flute":    dict(engine="flute", f0=440, pm=1.0, jet_ratio=0.24),
}

def play(name, dur=2.5):
    p = dict(PRESETS[name]); eng = p.pop("engine")
    if eng == "wind":
        rad, _ = wind(dur=dur, **p); sig = rad
    else:
        sig = flute(dur=dur, **p)
    print(name, "peaks:", find_peaks(sig[SR//4:])[:6])
    display(Audio(normalize(sig), rate=SR))
    return sig

for name in PRESETS:
    play(name)

## Tuning guide

**Getting a note in tune.** The pitch is set by the total delay length, but the loss
filter, radiation filter, DC blocker and interpolator all add a little group delay
that lengthens the loop and flattens the pitch. `loop_comp` (reed) / `length_comp`
(flute) subtracts that. Procedure: render, `find_peaks`, read the fundamental,
nudge the comp (bigger comp → shorter loop → sharper), repeat. A few Hz is usually
one sample of comp.

**When you change the radiation cutoff, re-tune.** A lower cutoff (bigger opening)
adds reflection group delay — the digital end correction — and flattens the note.
This coupling is physical: a real instrument's effective length depends on the
opening. Retrim `loop_comp` after changing `rad_cutoff_hz`.

**Changing pitch = changing length.** This model has **no tone holes**: every note
is a different total length (`f0`). Real woodwinds keep one tube and open holes to
shorten the effective length. Tone-hole modelling (a branched waveguide / register
vent) is a genuine future milestone, and it's also what would give a real clarinet
its twelfth-overblow and a flute its cross-fingerings.

**Timbre knobs, in order of effect.** Bore shape (odd vs full series) → source
sharpness (`reed_slope`, jet cubic) → radiation cutoff (bright vs full) → loss
(warm vs buzzy). Spend effort on the attack and the big spectral shape; detail finer
than a critical band is largely inaudible.

**Stability.** Every reflection `|r| ≤ 1` and every loss `< 1`, or the loop's energy
grows and it blows up. If you need to `clip` the output to stop a blow-up, that's a
diagnostic pointing at a loop-gain leak, not a fix.

## Honest limitations

Documented per the project's first-principles honesty rule:

* **Cone tuning.** The cylindrical-section cone fills in the even harmonics but its
  low modes are mistuned (truncated cone, no apex). The `sax_like` preset is a
  timbre demonstration, not an in-tune saxophone. A conical waveguide with an
  apex/mouthpiece correction is the real fix.
* **No faithful bell/horn.** The one-pole radiation end captures "highs escape, lows
  reflect" and the opening-size→cutoff relationship, but it does **not** reproduce a
  real bell's louder, evenly-projecting behaviour (a wider mouth here drains the loop
  and *quietens* it). A faithful horn needs the Webster horn equation and a
  frequency-dependent radiation impedance matched along the flare.
* **No tone holes.** Pitch is changed by changing total length, not by opening holes.
  This also means no register vent, so the clarinet gives threshold-and-choke rather
  than a spontaneous overblow to the twelfth.
* **Flute register sensitivity.** The jet flute's register depends on `jet_ratio`,
  breath and length together; the low-register window is real but narrow, and the
  presets are tuned points rather than a fully mapped playable range.
* **Simplified reed.** A single static reed table (clip), not a mass–spring reed with
  its own resonance — good for the clarinet's core behaviour, less so for fine
  articulation and altissimo.

None of these block the core result: correct woodwind *timbre* — odd-harmonic
clarinet, full-series flute, register behaviour, and the internal-vs-radiated
brightness — emerging from the physics rather than being dialled in.